# Giris

Bu proje, Kaggle uzerinde yer alan `House Prices - Advanced Regression Techniques` veri setini temel almaktadir.  
Veri, ABD’nin Iowa eyaletindeki Ames sehrinde satilan konutlara iliskin genis bir ozellik yelpazesi icermektedir.  
Bu calismada, evlerin satis fiyatlarini etkileyen faktorleri inceleyerek, bu fiyatlari en dogru sekilde tahmin edebilen bir model gelistirilmistir.


# Veri Setlerinin Yuklenmesi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score,GridSearchCV
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [ ]:
train=pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test=pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# Veri Hakkinda Bilgi

## Egitim Setindeki Hedef Degiskenin Analizi

Veriyi birlestirmeden onceden egitim setindeki `SalePrice` degiskeni hakkinda bilgi edinelim.

In [ ]:
sns.histplot(x='SalePrice',data=train,kde=True);

In [ ]:
len(train[train['SalePrice'] >= 500000])/len(train) #Egitim setinde 500.000 in uzerinde olan evlerin yuzdesi

In [ ]:
train[train['SalePrice'] >= 500000].sort_values(by='SalePrice', ascending=False)

Egitim setindeki `SalePrice` degiskeni grafigine(histogram) brakildiginda 500.000 in ustunde cok az evin oldugu gorulmektedir.
Bu evleri (kalite puani yasam alani buyuklugu) inceledigimizde ise `tipik` evlerin olmadigi aksina `ultraluks` grubuna girdigi gorulmektedir.

Bu yuzden modelin genelleme yetenegini artirmak icin veri setinin yuzde 0.001 ini bile olusturulmayan outlier degerler kaldirilmistir.



In [ ]:
train=train[train['SalePrice']<500000]
df=pd.concat([train,test],ignore_index=True)

Veri setindeki `Id` degiskeni her evin essiz numarasini temsil etmektedir dolayisiyla model icin hicbir bilgi icermediginden silinmelidir.

In [ ]:
df.drop('Id',axis=1,inplace=True)

In [ ]:
df.shape

**Degiskenler Hakkinda Bilgi**

1. `MSSubClass`: Satisa konu olan konutun tipini tanimlar (orn: 2 Katli 1946 ve sonrasi). [cite: 1]
2. `MSZoning`: Satisin genel imar siniflandirmasini tanimlar (orn: Konut Dusuk Yogunluk).
3. `LotFrontage`: Mulke bagli caddenin feet cinsinden uzunlugu.
4. `LotArea`: Parsel buyuklugu (feet kare).
5. `Street`: Mulke yol erisim tipi (Cakil veya Asfalt).
6. `Alley`: Ara sokaga erisim tipi (Cakil, Asfalt veya Yok). [cite: 5]
7. `LotShape`: Mulkun genel sekli (Duzgun, Az Duzensiz vb.).
8. `LandContour`: Mulkun duzlugu (Duz, Egimli vb.).
9. `Utilities`: Mevcut altyapi hizmetlerinin turu (Tum kamu hizmetleri vb.).
10. `LotConfig`: Parsel konfigurasyonu (Ic parsel, Kose parsel vb.).
11. `LandSlope`: Mulkun egimi (Hafif, Orta, Ciddi).
12. `Neighborhood`: Ames sehir sinirlari icindeki fiziksel konum (Semt).
13. `Condition1`: Cesitli kosullara yakinlik (Ana yol, Demiryolu vb.).
14. `Condition2`: Birden fazla kosul varsa ikincil kosul.
15. `BldgType`: Konut tipi (Mustakil, Dubleks vb.).
16. `HouseStyle`: Konutun stili (1 Katli, 2 Katli vb.).
17. `OverallQual`: Evin genel malzeme ve kaplama kalitesini derecelendirir (10 = Cok Mukemmel).
18. `OverallCond`: Evin genel durumunu derecelendirir (10 = Cok Mukemmel).
19. `YearBuilt`: Orijinal insaat tarihi.
20. `YearRemodAdd`: Tadilat tarihi (yoksa insaat tarihi ile ayni).
21. `RoofStyle`: Cati tipi (Kirma cati, Duz cati vb.).
22. `RoofMatl`: Cati malzemesi (Kompozit Singil vb.).
23. `Exterior1st`: Evdeki dis cephe kaplamasi.
24. `Exterior2nd`: Birden fazla malzeme varsa ikincil dis cephe kaplamasi.
25. `MasVnrType`: Duvar kaplama tipi (Tugla, Tas, Yok vb.).
26. `MasVnrArea`: Duvar kaplama alani (feet kare).
27. `ExterQual`: Dis cephedeki malzemenin kalitesini degerlendirir (Iyi, Ortalama vb.).
28. `ExterCond`: Dis cephedeki malzemenin mevcut durumunu degerlendirir.
29. `Foundation`: Temel tipi (Betonarme, Ahşap vb.).
30. `BsmtQual`: Bodrumun yuksekligini degerlendirir (Mukemmel: 100+ inch vb.).
31. `BsmtCond`: Bodrumun genel durumunu degerlendirir (Iyi, Tipik vb.).
32. `BsmtExposure`: Yuruyus veya bahce kati duvarlarini ifade eder (Iyi Pozlama, Pozlama Yok vb.). [cite: 22]
33. `BsmtFinType1`: Bodrumdaki bitmis alanin derecelendirmesi (Iyi Yasam Alani vb.).
34. `BsmtFinSF1`: Tip 1 bitmis alan (feet kare).
35. `BsmtFinType2`: Bodrumdaki (varsa) ikincil bitmis alanin derecelendirmesi.
36. `BsmtFinSF2`: Tip 2 bitmis alan (feet kare).
37. `BsmtUnfSF`: Bodrumun bitmemis alani (feet kare). [cite: 24]
38. `TotalBsmtSF`: Toplam bodrum alani (feet kare).
39. `Heating`: Isitma tipi (Gazli vb.).
40. `HeatingQC`: Isitma kalitesi ve durumu (Mukemmel, Iyi vb.).
41. `CentralAir`: Merkezi klima (Evet/Hayir).
42. `Electrical`: Elektrik sistemi (Standart Devre Kesiciler vb.).
43. `1stFlrSF`: Birinci kat alani (feet kare). [cite: 26]
44. `2ndFlrSF`: Ikinci kat alani (feet kare).
45. `LowQualFinSF`: Dusuk kaliteli bitmis alan (tum katlar, feet kare).
46. `GrLivArea`: Zemin ustu (yer ustu) yasam alani (feet kare).
47. `BsmtFullBath`: Bodrumdaki tam banyo sayisi.
48. `BsmtHalfBath`: Bodrumdaki yarim banyo sayisi.
49. `FullBath`: Zemin ustu tam banyo sayisi.
50. `HalfBath`: Zemin ustu yarim banyo sayisi.
51. `Bedroom`: Zemin ustu yatak odasi sayisi (bodrumdakiler dahil degil).
52. `Kitchen`: Mutfak sayisi.
53. `KitchenQual`: Mutfak kalitesi (Mukemmel, Iyi vb.).
54. `TotRmsAbvGrd`: Zemin ustu toplam oda sayisi (banyolar hariç).
55. `Functional`: Evin islevselligi (Tipik, Kucuk Kesintiler vb.). [cite: 27]
56. `Fireplaces`: Somine sayisi.
57. `FireplaceQu`: Somine kalitesi (Mukemmel, Iyi, Yok vb.).
58. `GarageType`: Garajin konumu (Eve Bitisik, Ayrı, Yok vb.).
59. `GarageYrBlt`: Garajin insa edildigi yil.
60. `GarageFinish`: Garajin ic bitis durumu (Bitmis, Yari Bitmis, Bitmemis, Yok).
61. `GarageCars`: Garajin arac kapasitesi.
62. `GarageArea`: Garajin alani (feet kare).
63. `GarageQual`: Garaj kalitesi (Iyi, Tipik, Yok vb.).
64. `GarageCond`: Garaj durumu (Iyi, Tipik, Yok vb.).
65. `PavedDrive`: Asfaltli araba yolu (Asfalt, Kismi, Toprak/Cakil).
66. `WoodDeckSF`: Ahşap guverte alani (feet kare).
67. `OpenPorchSF`: Acik veranda alani (feet kare). [cite: 31]
68. `EnclosedPorch`: Kapali veranda alani (feet kare).
69. `3SsnPorch`: Uc mevsimlik veranda alani (feet kare).
70. `ScreenPorch`: Sineklikli veranda alani (feet kare).
71. `PoolArea`: Havuz alani (feet kare).
72. `PoolQC`: Havuz kalitesi (Mukemmel, Iyi, Yok vb.).
73. `Fence`: Cit kalitesi (Iyi Gizlilik, Iyi Ahşap, Yok vb.).
74. `MiscFeature`: Diger kategorilere girmeyen ozellik (Asansor, Kulube, Yok vb.).
75. `MiscVal`: Diger ozelligin Dolar cinsinden degeri.
76. `MoSold`: Satildigi Ay (MM).
77. `YrSold`: Satildigi Yil (YYYY).
78. `SaleType`: Satis tipi (Normal, Yeni Insaat vb.).
79. `SaleCondition`: Satis kosulu (Normal, Anormal, Kismi vb.).

---


## Genel Resim

In [ ]:
def check_df(dataframe):
    print("##################### Shape #####################")
    print(dataframe.shape)
    print("##################### Types #####################")
    print(dataframe.dtypes)
    print("##################### Head #####################")
    print(dataframe.head(3))
    print("##################### Tail #####################")
    print(dataframe.tail(3))
    print("##################### NA #####################")
    print(dataframe.isnull().sum())
    print("##################### Quantiles #####################")
    print(dataframe.select_dtypes(include=[int,float]).quantile([0, 0.05, 0.50, 0.95, 0.99, 1]).T)


check_df(df)

In [ ]:
df[df['GarageYrBlt']==2207][['YearBuilt','YearRemodAdd']]

Burada eksik deger sayilarinin bazi degiskenlerde %90 ustu oldugu gorulmustur.

Sayisal sutunlardaki ceyrekliklere bakildiginda:
`GarageYrBlt` degiskenindeki en yuksek yilin 2207 yili oldugu gorulmektedir. Bu evin yapim yilina ve restore edilme yilina bakildiginda ise 2006 ve 2007 degerleri gorulmektedir, Burada evin 2006 da yapildigi garajin ise 2007 de yapildigi(2207 yazacagina buyuk ihtimal yazim hatasi yapmis) bunun yuzunden de restore edilme yilinin 2007 oldugu dusuncesine varabiliriz. Bu yuzden 2207 yili 2007 yili ile degistirilmelidir.

Sayisal degiskenler icin ceyrekliklere bakilmaya devam edildiginde ise cogu degiskenin saga carpik(bunu 0.99 ve 1.0 deki farktan anlayabilirsiniz) oldugu gorulmektedir. Bazi degiskenlerde ise gercekten asiri uc(LotArea,MiscVal,GrLivArea) degerlerin oldugu gorulmektedir. Bu degiskenlerde `feature engineering` den sonra log donusumu yapilmalidir.

In [ ]:
df.loc[df['GarageYrBlt'] == 2207,'GarageYrBlt']=2007

# Eksik Deger Analizi

In [ ]:
import missingno as msno

msno.matrix(df);

In [ ]:
def missing_values_table(dataframe, na_name=False):
    na_columns = [col for col in dataframe.columns if dataframe[col].isnull().sum() > 0]

    n_miss = dataframe[na_columns].isnull().sum().sort_values(ascending=False)

    ratio = (dataframe[na_columns].isnull().sum() / dataframe.shape[0] * 100).sort_values(ascending=False)

    missing_df = pd.concat([n_miss, np.round(ratio, 2)], axis=1, keys=['n_miss', 'ratio'])

    print(missing_df, end="\n")

    if na_name:
        return na_columns

na_name=missing_values_table(df,True)

In [ ]:
def check_missing(df, missing_col, value_col):
    print(f'{missing_col} eksik, ancak {value_col} 0 olmayan satir sayisi: {len(df[df[missing_col].isna() & (df[value_col] != 0)])}')


check_missing(df, 'PoolQC', 'PoolArea')
check_missing(df, 'MiscFeature', 'MiscVal')
check_missing(df, 'Fence', 'PoolArea')

Veri setimizdeki degisken sayisi cok fazla oldugu icin bazi degiskenleri silmede cesaretli olunabilir. Eksik deger yuzdeligine bakildiginda %80 in uzerinde olan 4 tane degisken gozukuyor.

<kbd>PoolQC degiskenindeki eksiklikler o havuzun olmadigi bilgisini verir. Havuzun olmadigi bilgisine PoolArea nin 0 oldugu bilgisinden de ulasabiliriz. Bu iliskiye uymayan sadece 3 tane satir var demek ki bu yaklasimimiz dogru.

<kbd>MiscFeature degiskenindeki eksiklikler diger kategorilere kapsanmayan ozelliklerin olmadigi bilgisini verir. MiscVal MiscFeatura da tanimlanan ozeligin degerini verir, MiscValdeki degerin 0 olmasi MiscFeature da bir ozelligin olmadigi bilgisini verebilir, Bu iliskiya uymayan sadece 1 tane satir var demek k ibu yaklasimimiz dogru.

**<kbd>Fence** Fence degiskeni cit bilgiisni PoolArea nin da 0 dan fazla olmasi havuz oldugu bilgisini verir. Gunluk hayattan eger bir havuz varsa citi de vardir ornegini bu veride test etmek istersek dogru oldugunu goruruz cunku bu iliskiye uymayan sadece 4 tane satir var.

**<kbd>Alley** degiskeni icin ara sokaga erisim tipini ifade eder. Buradaki eksik degerler onun yok oldugunu ifade eder. Alleydeki yok olanlarin sayisi yuzde 93 olacagindan oradaki eksik degerleri NaN ile doldurusak bir degiskendeki bariz ustunluk yuzunden modelde gurultu yaratabilir. O yuzden bu degisken icin eksik olan degerleri notna() yardimi ile doldurarak bu degiskeni binary degiskenine donusecek.


**<kbd>na_cat_cols** ve **<kbd>na_num_cols** degiskenlerinin icindeki ozellikler txt dosyasinda belirtildigi gibi aslinda `yok` ve `sifir` anlamindadir. Bu yuzden bu degiskenlere `fillna()` methodu uygulandi.

**<kbd>mode_cols** daki degiskenlerde ise sadece 1-4 satir eksik. Bunlar muhtemelen veri girisi hatasidir. Bu yuzden onlar direkt sayisallar median kategorikler de mode ile dolduruldu.



In [ ]:
drop_columns=['PoolQC','MiscFeature','Fence']
df=df.drop(drop_columns,axis=1)
df['Alley'] = df.pop('Alley').notna().astype(int)
na_cat_cols=['MasVnrType','FireplaceQu','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','GarageType','GarageFinish','GarageQual','GarageCond','Electrical']
df[na_cat_cols]=df[na_cat_cols].fillna('NaN')

na_num_cols=['MasVnrArea','GarageYrBlt','GarageCars','GarageArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath']
df[na_num_cols]=df[na_num_cols].fillna(0)

mode_cols=['MSZoning','Utilities','Exterior1st','Exterior2nd','KitchenQual','Functional','SaleType']
for col in mode_cols: 
    df[col]=df[col].fillna(df[col].mode()[0])

df['LotFrontage']=df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
df['LotFrontage']=df['LotFrontage'].fillna(df['LotFrontage'].median())

# Degisken Analizi

## Sayisal Degiksen Analizi

In [ ]:
num_cols=df.select_dtypes(include=[int,float]).columns
df[num_cols].hist(bins=20,figsize=(20,20));

In [ ]:
def num_summary(dataframe, numerical_col):
    quantiles = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]
    print(dataframe[numerical_col].describe(quantiles).T)

    print("#####################################")


for col in num_cols:
    num_summary(df, col)

Analiz

`MSSubClass`: Degerlerin yaklasik %30’u '20' olan bu degisken kategoriktir; sayisal degil, object (string) tipine donusturulmelidir.

`LotFrontage`: Medyani 70’tir; max=313 gibi asiri aykiri degerler vardir. Logaritmik donusum (np.log1p) uygulanmalidir.

`LotArea`: max=215245 gibi devasa aykiri degerler iceren, saga carpik bir degiskendir; logaritmik donusum (np.log1p) zorunludur.

`OverallQual`: Medyani 6 olan, dagilimi iyi, sirali (ordinal) bir degiskendir; oldugu gibi kullanilabilir.

`OverallCond`: Evlerin cogunun '5' (ortalama) degerini aldigi, degiskenligi dusuk sirali (ordinal) bir degiskendir; oldugu gibi kullanilabilir.

`YearBuilt`: Medyani 1973 olan, sola carpik (eski ev kuyrugu olan) bir degiskendir; “yas” (age) hesaplamasi icin kullanilmalidir.

`YearRemodAdd`: Medyani 1993 olan, sola carpik bir degiskendir; “yas” (age) hesaplamasi icin kullanilmalidir.

`MasVnrArea`: Degerlerin %60’i sifir olan (sifir-agirlikli) ve saga carpik bir degiskendir; np.log1p uygulanmalidir.

`BsmtFinSF1`: Degerlerin %30’u sifir olan, max=5644 gibi buyuk aykiri degerler iceren saga carpik bir degiskendir; np.log1p zorunludur.

`BsmtFinSF2`: Degerlerin %80’i sifirdir; SalePrice uzerinde zayif negatif sinyali oldugu icin NEW_TotalBsmtFin icinde birlestirilmelidir.

`BsmtUnfSF`: Medyani 467 olan, saga carpik bir degiskendir; np.log1p uygulanmalidir.

`TotalBsmtSF`: max=6110 gibi asiri aykiri degerler iceren, saga carpik bir degiskendir; np.log1p zorunludur.

`1stFlrSF`: max=5095gibi asiri aykiri degerler iceren, saga carpik bir degiskendir; np.log1p zorunludur.

`2ndFlrSF`: Degerlerin %40-50’si sifir olan (tek katli evler), saga carpik bir degiskendir; np.log1p uygulanmalidir.

`LowQualFinSF`: Degerlerin %95’i sifirdir; SalePrice uzerinde negatif sinyali oldugu icin (analizimize gore) silinmemeli, HasLowQual (1/0) degiskenine donusturulmelidir.

`GrLivArea`: Saga carpik, cok onemli bir degiskendir; EGITIM SETINDEKI 2 anormal aykiri satiri (ID 521, 1289) SILINMELI (drop edilmeli) ve kalan sutuna np.log1p uygulanmalidir.

`BsmtFullBath`, `FullBath`, `HalfBath`: Bu dort ayri banyo sayma degiskeni birlestirilmelidir.

`BedroomAbvGr`: Medyani 3 olan, max=8 gibi aykiri degerler icerebilen, dagilimi normal bir degiskendir.

`KitchenAbvGr`: Degerlerin %95’i '1' olan, cok dusuk degiskenlige sahip bu degisken; silinebilir veya binary (1/0) degiskene donusturulebilir.

`TotRmsAbvGrd`: Medyani 6 olan, dagilimi normal bir degiskendir ancak GrLivArea ile yuksek korelasyon gosterebilir.

`Fireplaces`: Evlerin yaklasik %40-50’sinde 0 (somine yok), kalanlarin cogunda 1 vardir.

`GarageYrBlt`, `GarageCars`, `GarageArea`: Bu uc degiskenin her biri %5-%10 oraninda sifir-agirliklidir (garaj yok). GarageCars ile GarageArea yuksek korelasyonludur.

`WoodDeckSF`, `OpenPorchSF`, `EnclosedPorch`, `3SsnPorch`, `ScreenPorch`: Tamami sifir-agirliklidir; SalePrice analizimize gore hepsi sinyal tasir. TotalPorchSF icinde birlestirilmeli ve/veya Has... (1/0) degiskenlerine donusturulmelidirler.

`PoolArea`: Degerlerin %99’u sifirdir; SalePrice uzerinde guclu pozitif sinyali oldugu icin HasPool (1/0) degiskenine donusturulmelidir.

`MiscVal`: Degerlerin %95’i sifirdir; SalePrice uzerinde negatif sinyali oldugu icin HasMiscFeature (1/0) degiskenine donusturulmelidir.

`MoSold`: 1-12 arasindaki aylarin temsilcisi; sayisal degil, kategorik (object) tipine donusturulmeli diye dusunulebilir ama siniflar arasindaki hedef degiskenlerin ortalamasina bakildiginda aradaki fark cok az oldugu icin silinmelidir.

`YrSold`: 2006-2010 yillari arasini kapsar; “yas” (age) degiskenleri turetmek icin kullanilmalidir.

`SalePrice`: count=1451 olan, saga carpik hedef degiskenimizdir; logaritmik donusum (np.log1p) zorunludur.

`Alley`: Evlerin sadece %6.8’inde bulunmaktadir; modele hazir (1/0) binary bir degiskendir.


In [ ]:
print(f"MoSold degiskeninin siniflarinin hedef degisken uzerindeki analizi:\n{df.groupby('MoSold')['SalePrice'].mean()}")

Ayrica bazi cok yuksek aykiri degerler vardir bu aykiri degerleri log donusumunde biraz baskilayabiliriz sonra hala aykiriligi cok fazla devam ediliyorsa silinme islemi uygulanabilir ama **<kbd>GrLivArea** daki asiri uc degerlerdeki satirlara bakildiginda bazi tutarsizliklar gozlemlenmektedir.

In [ ]:
df[df['GrLivArea']>4500]

In [ ]:
df[df['SaleCondition']=='Partial']['SalePrice'].mean()

Bu evler veri setindeki en buyuk ve en kaliteli(OverallQual=10) evler olmasina ragmen satis fiyatlari 160-184k anormal derecede dusuktur.
Bu evlerin neden boyle olduguna baktigimizda ise tamamlanmamis ev oldugunu goruyoruz ama tamamlanmamis evlerin ortalamarina bakildiginda ise bu evlerin fiyatindan yine oldukca yuksek(262k) oldugunu gorulmektedir.

**Sonuc**

Bu evleri silmeliyiz cunku `en buyuk ve en kaliteli evler ortalama bir ev fiyatina satilir` gibi tamamen yanlis ve celiskili bir bilgi ogretir.

In [ ]:
outlier_indices = df[(df['GrLivArea'] > 4500) & (df['SalePrice'].notna())].index
df = df.drop(outlier_indices)

In [ ]:
df.drop('MoSold',axis=1,inplace=True)

## Kategorik Degisken Analizi

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def cat_summary_with_target(dataframe, col_name, target, plot=False):
    summary=pd.DataFrame({
        'Count': dataframe[col_name].value_counts(),
        'Ratio': 100 * dataframe[col_name].value_counts() / len(dataframe),
        'TARGET_MEAN': dataframe.groupby(col_name)[target].mean()
    })
    print(summary, end="\n\n\n")

cat_cols=[i for i in df.columns if i not in num_cols]

for col in cat_cols:
    cat_summary_with_target(df,col,'SalePrice')


1. Direkt OHE (One-Hot Encoding) Yapilacaklar

Bu degiskenler, kategorileri arasinda 'Iyi > Kotu' gibi net bir siralama (hiyerarsi) bulunmadigi ve kategorileri rare_map (nadir birlestirme) planimiza dahil edecek kadar dengesiz veya nadir olmadigi icin OHE ile kodlanacaktir.

`MSZoning`: Cunku bolge siniflamasi ('RL', 'RM' vb.) sirali degildir.

`Street`: Cunku yol tipi ('Pave', 'Grvl') sirali degildir.

`LotShape`: Cunku parsel sekli ('Reg', 'IR1' vb.) sirali degildir.

`LandContour`: Cunku arazi duzlugu ('Lvl', 'Bnk' vb.) sirali degildir.

`Utilities`: Cunku altyapi tipleri ('AllPub', 'NoSeWa') sirali degildir.

`LotConfig`: Cunku parsel konumu ('Inside', 'Corner' vb.) sirali degildir.

`Neighborhood`: Cunku semt isimleri ('CollgCr', 'Veenker' vb.) sirali degildir.

`Condition1`: Cunku kosul ('Norm', 'Feedr' vb.) sirali degildir.

`BldgType`: Cunku bina tipi ('1Fam', 'Duplex' vb.) sirali degildir.

2. Once Kategoriler Birlestirilecek, Sonra OHE Yapilacaklar

Bu degiskenler de sirali degildir, ancak bazi kategorileri asiri nadir, anlamsiz gurultu yaratan veya fiyat ortalamasi olarak benzer baska bir kategoriye dahil edilebilecek (son analizlerimize gore) durumdaydi. Bu yuzden gurultuyu azaltmak icin once bu kategorileri birlestirip, sonra OHE yapacagiz.

`LandSlope`: Cunku 'Gtl' (%95) baskin oldugu icin, nadir olan 'Mod' ve 'Sev' kategorilerini 'Rare' olarak birlestirdik.

`Condition2`: Cunku 'Norm' (%99) asiri baskin oldugu icin, kalan tum nadir kategorileri 'Rare' olarak birlestirdik.

`HouseStyle`: Cunku asiri nadir '2.5Fin' kategorisini, mantiksal benzeri olan '2Story' ile birlestirdik.

`RoofStyle`: Cunku 'Gable' ve 'Hip' disindaki nadir kategoriler olan 'Flat' ve 'Mansard'i 'Rare' olarak birlestirdik.

`RoofMatl`: (Guncel Karar) Cunku fiyat analiziyle, fiyati yakin 'Tar&Grv'yi ana kategori 'CompShg' ile; kalan her seyi 'Rare_Roof' olarak birlestirdik.

`Exterior1st`: Cunku nadir kategorileri fiyat ortalamalarina gore 'Rare' ve 'Rare2' olarak grupladik.

`Exterior2nd`: Cunku nadir kategorileri fiyat ortalamalarina gore 'Rare' ve 'Rare2' olarak grupladik.

`MasVnrType`: Cunku nadir 'BrkCmn' kategorisini, benzer fiyatli 'NA' (None/Yok) kategorisi ile birlestirdik.

`Foundation`: Cunku nadir olan 'Stone' ve 'Wood' kategorilerini 'Rare' olarak birlestirdik.

`Heating`: Cunku 'GasA' (%98.4) asiri baskin oldugu icin, kalan tum nadir ve dusuk fiyatli kategorileri 'Rare' olarak birlestirdik.

`Electrical`: Cunku 'SBrkr' baskin oldugu icin, asiri nadir 'FuseP' ve 'Mix' kategorilerini 'Rare' olarak birlestirdik.

`GarageType`: Cunku nadir olan '2Types' ve 'Basment' kategorilerini 'Rare' olarak birlestirdik.

`SaleType`: Cunku 'WD' disindaki nadir kategorileri, fiyat analizimize gore 'Rare', 'Rare2' ve 'Rare3' olarak grupladik.

`SaleCondition`: Cunku fiyat analizimize gore 'Alloca'yi 'Normal' ile; 'Abnorml' ve 'Family'yi ise 'Rare' olarak grupladik.

3. Ordinal Encoding (Sirali Kodlama) Yapilacaklar

Bu degiskenler, 'Mukemmel > Iyi > Kotu' gibi net bir siralama (hiyerarsi) icerir. OHE yapmak bu onemli siralama bilgisini yok edecegi icin, bu degiskenlere (5, 4, 3...) gibi sayisal degerler atayan Ordinal Encoding yapacagiz.

`ExterQual`: Cunku dis cephe kalitesini ('Ex' > 'Gd') siralar.

`ExterCond`: Cunku dis cephe durumunu ('Ex' > 'Gd') siralar.

`BsmtQual`: Cunku bodrum kalitesini ('Ex' > 'Gd') ve yoklugunu ('NA'=0) siralar.

`BsmtCond`: Cunku bodrum durumunu ('Ex' > 'Gd') ve yoklugunu ('NA'=0) siralar.

`BsmtExposure`: Cunku bodrum pozlamasini ('Gd' > 'Av' > 'No') ve yoklugunu ('NA'=0) siralar.

`BsmtFinType1`: Cunku bodrum bitis kalitesini ('GLQ' > 'ALQ' > 'Unf') ve yoklugunu ('NA'=0) siralar.

`BsmtFinType2`: Cunku bodrum 2. bitis kalitesini ('GLQ' > 'ALQ' > 'Unf') ve yoklugunu ('NA'=0) siralar.

`HeatingQC`: Cunku isitma kalitesini ('Ex' > 'Gd') siralar.

`CentralAir`: Cunku bu binary bir degiskendir ('Y' > 'N'), bu nedenle (Y=1, N=0) olarak siralanir.

`KitchenQual`: Cunku mutfak kalitesini ('Ex' > 'Gd') siralar.

`FireplaceQu`: Cunku somine kalitesini ('Ex' > 'Gd') ve yoklugunu ('NA'=0) siralar.

`GarageFinish`: Cunku garaj bitis durumunu ('Fin' > 'RFn' > 'Unf') ve yoklugunu ('NA'=0) siralar.

`GarageQual`: Cunku garaj kalitesini ('Ex' > 'Gd') ve yoklugunu ('NA'=0) siralar.

`GarageCond`: Cunku garaj durumunu ('Ex' > 'Gd') ve yoklugunu ('NA'=0) siralar.

`PavedDrive`: Cunku yol durumunu ('Y' > 'P' > 'N') siralar.

4. Once Birlestirilen, Sonra Ordinal Kodlanan (Hibrit)

Bu degisken icin once gurultuyu azaltmak amaciyla nadir kategorileri birlestirdik, ardindan kalan temiz kategorilere mantikli bir siralama uyguladik.

`Functional`: Cunku once 'Min1', 'Min2' gibi nadir kategoriler 'Rare' olarak gruplandi, sonra ('Typ' > 'Rare' > 'Mod'...) seklinde sirali olarak kodlandi.


# Encoding  Islemleri

## Kategorik ama Sayisal Sutunleri Object Veri Tipine Donusturme

In [ ]:
df['MSSubClass']=df['MSSubClass'].astype(str)

## Rare Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
rare_map = {
    'LandSlope':{'Rare':['Mod','Sev']},
    'HouseStyle':{'2Story':['2.5Fin']},
    'Condition2':{'Rare':['Artery','Feedr','RRAe','RRAn','RRNe']},
    'RoofStyle':{'Rare':['Flat','Mansard']},
    'RoofMatl':{'CompShg':['Tar&Grv'],'Rare_Roof':['Membran','Metal','Roll','WdShake','WdShngl','ClyTile']},
    'Exterior1st':{'Rare':['AsphShn','CBlock','AsbShng'],'Rare2':['Stucco','WdShing']},
    'Exterior2nd':{'Rare':['CBlock','AsphShn','Brk Cmn'],'Rare2':['Stone','Wd Shng']},
    'MasVnrType':{'Rare':['BrkCmn']},
    'Foundation':{'Rare':['Stone','Wood']},
    'Heating':{'Rare':['Floor','Grav','OthW','Wall']},
    'Electrical':{'Rare':['FuseP','Mix']},
    'Functional':{'Rare':['Min1','Min2','Maj1']},
    'GarageType':{'Rare':['2Types','Basment']},
    'SaleType':{'Rare':['New','Con'],'Rare2':['ConLI','CWD'],'Rare3':['ConLD','ConLw','Oth']},
    'SaleCondition':{'Rare':['Abnorml','Family'],'Normal':['Alloca']},
}
df['Street'] = LabelEncoder().fit_transform(df['Street'])
df.drop(['Utilities'],axis=1,inplace=True)

for col, mapping in rare_map.items():
    replace_dict={}
    for new_val,old_vals in mapping.items():
        for old in old_vals:
            replace_dict[old]=new_val
    df[col]=df[col].replace(replace_dict)




## Ordinal Encoding

In [ ]:
qual_map_standard={'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'NaN':0}
bsmt_exposure_map={'Gd':4,'Av':3,'Mn':2,'No':1,'NaN':0}
bsmt_fin_map={'GLQ':6,'ALQ':5,'BLQ':4,'Rec':3,'LwQ':2,'Unf':1,'NaN':0}
garage_finish_map={'Fin':3,'RFn':2,'Unf':1,'NaN':0}
paved_drive_map={'Y':2,'P':1,'N':0}
functional_map={'Typ':7,'Rare':5,'Mod':4,'Maj2':2,'Sev':1,'Sal':0}
central_air_map={'Y':1,'N':0}
neighborhood_map = {
    'NoRidge': 6, 'NridgHt': 6, 'StoneBr': 6,
    'Timber': 5, 'Veenker': 5, 'Somerst': 5,
    'Crawfor': 5, 'ClearCr': 5,
    'CollgCr': 4, 'Blmngtn': 4, 'Gilbert': 4, 'NWAmes': 4,
    'SawyerW': 4,
    'Mitchel': 3, 'NAmes': 3,
    'NPkVill': 2, 'SWISU': 2, 'Sawyer': 2,
    'OldTown': 2, 'Edwards': 2,
    'BrkSide': 1, 'BrDale': 1, 'IDOTRR': 1, 'MeadowV': 1, 'Blueste': 1
}



qual_cols_standard=['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual','FireplaceQu','GarageQual','GarageCond']

for col in qual_cols_standard: 
    df[col]=df[col].replace(qual_map_standard)
df['BsmtExposure']=df['BsmtExposure'].replace(bsmt_exposure_map)
df['BsmtFinType1']=df['BsmtFinType1'].replace(bsmt_fin_map)
df['BsmtFinType2']=df['BsmtFinType2'].replace(bsmt_fin_map)
df['Functional']=df['Functional'].replace(functional_map)
df['GarageFinish']=df['GarageFinish'].replace(garage_finish_map)
df['PavedDrive']=df['PavedDrive'].replace(paved_drive_map)
df['CentralAir']=df['CentralAir'].replace(central_air_map)
df['Neighborhood']=df['Neighborhood'].replace(neighborhood_map)


## Feature Engineering

`NoGarage`: Bir evin garajinin olup olmadigini (1/0) belirlemek icin olusturuldu. Garaj yoklugu (GarageType == 'NaN'), analizlerimize gore fiyati dusuren onemli bir faktordu.

`NoModernElectric`: Bir evin modern elektrik sistemine ('SBrkr') sahip olup olmadigini (1/0) belirlemek icin olusturuldu. Eski sistemler (Electrical != 'SBrkr') daha dusuk fiyatla iliskilidir.

`Condition_any_norm`: Evin birincil veya ikincil kosulunun 'Normal' olup olmadigini (1/0) belirlemek icin olusturuldu. 'Normal' disi kosullar (yakin yollar, demiryolu) fiyati etkileyen bir sinyaldir.

`HouseAge`: Evin satildigi tarihteki yasini (YrSold - YearBuilt) hesaplamak icin olusturuldu. Evin yasi, fiyati belirleyen en onemli faktorlerden biridir.

`AgeSinceRemod`: Evin satildigi tarihte, en son tadilattan bu yana kac yil gectigini (YrSold - YearRemodAdd) olcmek icin olusturuldu. Yeni tadilat fiyati artirir.

`IsRemodeled`: Evin hic tadilat gorup gormedigini (1/0) belirlemek icin olusturuldu (YearRemodAdd != YearBuilt). Tadilat gormus olmasi pozitif bir sinyaldir.

`NEW_Restoration`: Evin insaati ile ilk tadilati arasinda kac yil gectigini (YearRemodAdd - YearBuilt) olcmek icin olusturuldu.

`NEW_GarageSold`: Garajin satildigi tarihteki yasini (YrSold - GarageYrBlt) hesaplamak icin olusturuldu.

`TotalBath`: 4 farkli banyo sutununu (FullBath, HalfBath, BsmtFullBath, BsmtHalfBath) tek ve daha guclu bir 'Toplam Banyo Sayisi' ozelliginde birlestirmek icin olusturuldu.

`TotalPorchSF`: 5 farkli veranda/teras tipini (OpenPorchSF, WoodDeckSF vb.) tek ve daha guclu bir 'Toplam Veranda Alani' ozelliginde birlestirmek icin olusturuldu.

`TotalSF`: Bir evin toplam yasanabilir alanini (GrLivArea + TotalBsmtSF) hesaplamak icin olusturuldu. Bu, fiyati tahmin etmedeki en guclu degiskenlerden biridir.

`NEW_TotalFlrSF`: Zemin katlarin toplam alanini (1stFlrSF + 2ndFlrSF) bulmak icin olusturuldu.

`NEW_TotalBsmtFin`: Iki farkli bitmis bodrum alani tipini (BsmtFinSF1 + BsmtFinSF2) tek bir 'Toplam Bitmis Bodrum Alani' ozelliginde birlestirmek icin olusturuldu.

`NEW_OverallGrade`: Iki farkli sirali kalite olcumunu (OverallQual * OverallCond) carparak tek ve daha guclu bir 'Genel Puan' ozelligi yaratmak icin olusturuldu.

`GarageAreaPerCar`: GarageArea ve GarageCars arasindaki yuksek korelasyonu (multicollinearity) cozmek ve arac basina dusen alani (verimliligi) olcmek icin olusturuldu.

`NEW_LotRatio`: Evin yasam alaninin (GrLivArea), toplam arsa alanina (LotArea) oranini bulmak icin olusturuldu. Bu, arsanin ne kadarinin kullanildigini gosterir.

`NEW_GarageLotRatio`: Garaj alaninin (GarageArea), toplam arsa alanina (LotArea) oranini bulmak icin olusturuldu.

`HasPool`: PoolArea sutununun %99'u sifirdi. Analizimiz (17:48), havuz varliginin (1/0) fiyati guclu bir sekilde artirdigini gosterdi. Bu sinyali yakalamak icin olusturuldu.

`HasMiscFeature`: MiscVal sutununun %95'i sifirdi. Analizimiz (17:48), bu ozelligin varliginin (1/0) fiyati dusurdugunu gosterdi. Bu negatif sinyali yakalamak icin olusturuldu.

`HasLowQual`: LowQualFinSF sutununun %95'i sifirdi. Analizimiz (17:48), bu ozelligin varliginin (1/0) fiyati dusurdugunu gosterdi. Bu negatif sinyali yakalamak icin olusturuldu.

`Has_Nothing`: Hicbir temel ozellige (klima yok, bodrum yok, garaj yok, modern elektrik yok vb.) sahip olmayan, en dusuk kaliteli evleri (1/0) etiketleyerek guclu bir negatif sinyal yakalamak icin olusturuldu.


In [ ]:
import pandas as pd
import numpy as np

df['NoGarage']=(df['GarageType']=='NaN').astype(int)
df['NoModernElectric']=(df['Electrical']!='SBrkr').astype(int)
df['Condition_any_norm']=((df['Condition1']=='Norm')|(df['Condition2']=='Norm')).astype(int)
df['HouseAge']=df['YrSold']-df['YearBuilt']
df['AgeSinceRemod']=df['YrSold']-df['YearRemodAdd']
df['IsRemodeled']=(df['YearRemodAdd']!=df['YearBuilt']).astype(int)
df["NEW_Restoration"]=df['YearRemodAdd']-df['YearBuilt']
df["NEW_GarageSold"]=df['YrSold']-df['GarageYrBlt']
df['TotalBath']=df['FullBath']+0.5*df['HalfBath']+df['BsmtFullBath']+0.5*df['BsmtHalfBath']
df['TotalPorchSF']=df['OpenPorchSF']+df['EnclosedPorch']+df['WoodDeckSF']+df['3SsnPorch']+df['ScreenPorch']
df["TotalSF"]=df["GrLivArea"]+df["TotalBsmtSF"]
df["NEW_TotalFlrSF"]=df["1stFlrSF"]+df["2ndFlrSF"]
df["NEW_TotalBsmtFin"]=df['BsmtFinSF1']+df['BsmtFinSF2']
df["NEW_OverallGrade"]=df['OverallQual']*df['OverallCond']
df['GarageAreaPerCar']=df.apply(lambda x:x['GarageArea']/x['GarageCars'] if x['GarageCars']>0 else 0,axis=1)
df["NEW_LotRatio"]=df['GrLivArea']/(df['LotArea']+1e-6)
df["NEW_GarageLotRatio"]=df['GarageArea']/(df['LotArea']+1e-6)
df['HasPool']=(df['PoolArea']>0).astype(int)
df['HasMiscFeature']=(df['MiscVal']>0).astype(int)
df['HasLowQual']=(df['LowQualFinSF']>0).astype(int)
df['Has_Nothing']=(df['CentralAir']==0)&(df['BsmtQual']==0)&(df['FireplaceQu']==0)&(df['PavedDrive']==0)&(df['NoGarage']==1)&(df['NoModernElectric']==1)
df['Has_Nothing']=df['Has_Nothing'].astype(int)


# Bazi Degiskenlerin Silinip Silinmeyecegine Karar Verilmesi

Olusturudugumuz yeni degiskenlerde bazi degiskenler kullandik. Simdi eski degiskenlerin durumunun ne olacagina karar verecegiz. Ilk olarak kolerasyonlara bakalim.

In [ ]:
num_df=df.select_dtypes(include=[int,float])
sns.heatmap(num_df.corr(), annot=False, cmap='coolwarm', center=0)

In [ ]:
num_cols=df.select_dtypes(include=[int,float])
print(num_df.corr()['SalePrice'].sort_values(ascending=False))

In [ ]:
import pandas as pd
import numpy as np

analysis_groups={
    'Alan_Grubu':['TotalSF','GrLivArea','TotalBsmtSF'],
    'Kat_Grubu':['NEW_TotalFlrSF','1stFlrSF','2ndFlrSF'],
    'Yas_Grubu':['HouseAge','YearBuilt','AgeSinceRemod','YearRemodAdd'],
    'Garaj_Alani_Grubu':['GarageArea','GarageCars'],
    'Genel_Kalite_Grubu':['NEW_OverallGrade','OverallQual','OverallCond'],
    'Banyo_Grubu':['TotalBath','FullBath','BsmtFullBath','HalfBath','BsmtHalfBath'],
    'Veranda_Grubu':['TotalPorchSF','OpenPorchSF','WoodDeckSF','EnclosedPorch']
}

for group_name, cols in analysis_groups.items():
    available_cols=[col for col in cols if col in num_df.columns]
    print(f'{num_df[available_cols].corr()}\n\n')

In [ ]:
df_numeric = df.select_dtypes(include=[int,float])
df_numeric = df_numeric.drop(columns=['SalePrice'])

corr_pairs = df_numeric.corr().unstack()
sorted_pairs_index = corr_pairs.index.map(lambda x: tuple(sorted(x)))
corr_pairs = corr_pairs[~sorted_pairs_index.duplicated()]
corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1)]

print('En guclu ilk 10 kolerasyon(ikili)')
print(corr_pairs.sort_values(ascending=False).head(10))
print('\n\nEn dusuk ilk 10 kolerasyon(ikili)')
print(corr_pairs.sort_values(ascending=True).head(10))


**Sonuc**

1. Daha Guclu Yeni Degiskenlerle Degistirilenler

Bu sutunlarin tasidigi bilgi, yeni olusturdugumuz (ve SalePrice ile korelasyonu daha yuksek olan) degiskenlere aktarildigi icin silinmistir.

`GrLivArea`, `TotalBsmtSF`: Cunku bilgileri, korelasyonu daha guclu olan `TotalSF` degiskeninde birlestirildi.

`YearBuilt`, `YearRemodAdd`, `YrSold`, `GarageYrBlt`: Cunku bilgileri, korelasyonu daha guclu olan `HouseAge`, `AgeSinceRemod` ve `NEW_GarageSold` degiskenlerine aktarildi.

`FullBath`, `HalfBath`, `BsmtFullBath`, `BsmtHalfBath`: Cunku bilgileri, daha guclu bir sinyal olan `TotalBath` degiskeninde birlestirildi.

`OpenPorchSF`, `EnclosedPorch`, `WoodDeckSF`, `3SsnPorch`, `ScreenPorch`: Cunku bilgileri, daha guclu bir sinyal olan `TotalPorchSF` degiskeninde birlestirildi.

`BsmtFinSF1`, `BsmtFinSF2`: Cunku bilgileri `NEW_TotalBsmtFin` degiskeninde birlestirildi.

`1stFlrSF`, `2ndFlrSF`: Cunku bilgileri `NEW_TotalFlrSF` degiskeninde birlestirildi.

`LotArea`: Cunku `NEW_LotRatio` gibi oran degiskenlerini olusturmak icin kullanildi ve asiri carpiticigi vardi.

`Condition1`, `Condition2`: Cunku bilgileri, daha basit bir sinyal olan `Condition_any_norm` (1/0) degiskenini olusturmak icin kullanildi.

2. Coklu Baglanti (Multicollinearity) Nedeniyle Silinenler

Bu sutunlar, modelde tuttugumuz baska bir sutunla asiri yuksek korelasyona (%85+) sahip olduklari icin silinmistir.

`GarageCars`, `GarageArea`: Cunku bu iki degisken birbiriyle asiri yuksek korelasyona (0.892) sahipti (Garaj buyuklugunu iki kez olcmemek icin).

`Fireplaces`: Cunku `FireplaceQu` (Ordinal Kalite) ile asiri yuksek korelasyona (0.863) sahipti ve SalePrice ile sinyali daha zayifti.

3. Sinyali 1/0 (Binary) Degiskene Aktarilanlar

Bu sutunlarin neredeyse tamami sifirdi ve tasidiklari sinyal (Var/Yok) yeni 1/0 degiskenlerine aktarildi.

`PoolArea`: Cunku sinyali (Havuz Varligi) `HasPool` (1/0) degiskenine aktarildi.

`MiscVal`: Cunku negatif sinyali `HasMiscFeature` (1/0) degiskenine aktarildi.

`LowQualFinSF`: Cunku negatif sinyali `HasLowQual` (1/0) degiskenine aktarildi.

4. Basarisiz Ozellik Muhendisligi Oldugu Icin Silinen

Bu, bizim olusturdugumuz ancak orijinalinden daha kotu performans gosteren bir degiskendir.

`NEW_OverallGrade`: Cunku bu yeni degiskenin SalePrice korelasyonu (0.574), orijinal `OverallQual`'in (0.807) korelasyonundan daha dusuk cikti


In [ ]:
cols_to_drop = [
    'YrSold','YearBuilt','YearRemodAdd','GarageYrBlt',
    'FullBath','HalfBath','BsmtFullBath','BsmtHalfBath',
    'GrLivArea','TotalBsmtSF','1stFlrSF','2ndFlrSF',
    'BsmtFinSF1','BsmtFinSF2','LotArea','GarageArea',
    'OpenPorchSF','EnclosedPorch','WoodDeckSF','3SsnPorch','ScreenPorch',
    'GarageCars','Condition1','Condition2',
    'PoolArea','MiscVal','LowQualFinSF',
    'Fireplaces','NEW_OverallGrade','MoSold'
]

df = df.drop(columns=cols_to_drop, errors='ignore')


# Veriyi Modele Hazirlama

## Sayisal Degiskenler icin Carpikligi Duzeltme

**1-Logaritma Donusumu Gerektirmeyen Degiskenler**  

Ordinal Kodlananlar: `OverallQual` `OverallCond` `ExterQual` `ExterCond` `BsmtQual` `BsmtCond` `BsmtExposure` `BsmtFinType1` `BsmtFinType2` `HeatingQC` `KitchenQual` `FireplaceQu` `GarageFinish` `GarageQual` `GarageCond` `PavedDrive` `Functional`  

Binary Kodlananlar (0/1): `CentralAir` `Alley` `NoGarage` `NoModernElectric` `Condition_any_norm` `IsRemodeled` `HasPool` `HasMiscFeature` `HasLowQual` `Has_Nothing`  

Gerçek Sayma Değişkenleri (Count): `TotalBath` `BedroomAbvGr` `KitchenAbvGr` `TotRmsAbvGrd`  


**2-Logaritma Donusumu Gerektiren Degiskenler** `LotFrontage` `MasVnrArea` `BsmtUnfSF` `HouseAge` `AgeSinceRemod` `NEW_Restoration` `NEW_GarageSold` `TotalPorchSF` `TotalSF` `NEW_TotalFlrSF` `NEW_TotalBsmtFin` `GarageAreaPerCar` `NEW_LotRatio` `NEW_GarageLotRatio`


Modelleme oncesi olusturudugumuz `feature` larda hata var mi ona bakilmasi gerekmektedir.

In [ ]:
log_cols=['LotFrontage','MasVnrArea','BsmtUnfSF','HouseAge','AgeSinceRemod',
          'NEW_Restoration','NEW_GarageSold','TotalPorchSF','TotalSF',
          'NEW_TotalFlrSF','NEW_TotalBsmtFin','GarageAreaPerCar','NEW_LotRatio',
          'NEW_GarageLotRatio']
neg_cols=[]
for col in log_cols:
    if df[col].min()<0:
        neg_cols.append(col)
        print(col,df[col].min())

Burada ilk olarak dusunulen sey acaba degiskenlerin yerinin karisitirlip karistirilmadigidir. Ama burada hangi iki degikseni yer degistirirsek degistirelim bu satirlarda en az bir negatif deger kalcaktir. Bu yuzden en mantikli cozum negatif degerlere sahip olanlari 0 a esitlemektir.

In [ ]:
for col in neg_cols:
    df[col]=df[col].clip(lower=0)

In [ ]:
print('Kontrol')
df[(df['HouseAge']<0) | (df['AgeSinceRemod']<0) | (df['NEW_Restoration']<0) | (df['NEW_GarageSold']<0)]

In [ ]:
for col in log_cols:
    num_summary(df, col)

Bazı `NEW_GarageSold` degerleri 2000'li gibi gozukuyor, bu evde garaj olmadigini gosteriyor. Logaritma donusumu uygulandiginda, bu buyuk degerler diger garaj yaslariyla uyumlu hale gelir ve model icin guvenli olur.(Logaritma donusumu icin sectigimiz degiksenlerin carpikliginin yuzde 75in ustunde olup olmadigini kontrol etmeliyiz. Ustunde olanlara log donusumu yapacagiz.)

In [ ]:
for col in log_cols:
    if abs(df[col].skew()) <=0.75:
        df[col]=np.log1p(df[col])

## OHE 

Bu değişkenlerin hiçbiri sıralı değil, hepsi kategorik (nominal) olduğundan, modelde doğru temsil için One-Hot Encoding (OHE) uygulanmalıdır.

In [ ]:
cat_cols=df.select_dtypes(include='O').columns
df=pd.get_dummies(df,columns=cat_cols,drop_first=True)


# Modeli Olusturma

Bu modelde hiperparametre optimizasyonu icin GridSearchCV kullanilmistir. 
Ancak, denenen parametre araliklari sonucunda elde edilen skor, temel modelden daha dusuk cikmistir. 
Bu nedenle, model karmaşasını artirmamak adina GridSearch sonuclari iptal edilmis ve orijinal ayarlar korunmustur.

In [ ]:
train_df=df[df['SalePrice'].notnull()]
test_df_X=df[df['SalePrice'].isnull()].drop('SalePrice',axis=1)
y=train_df[['SalePrice']] 
X=train_df.drop("SalePrice", axis=1)
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.2,random_state=1907)
y_train,y_val,y=np.log1p(y_train),np.log1p(y_val),np.log1p(y)

In [ ]:
model=CatBoostRegressor(random_state=72,verbose=False)
model.fit(X_train,y_train)
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val,y_pred))
print(f"CatBoost RMSE: {rmse:.4f}")

In [ ]:
y_pred=np.exp(model.predict(test_df_X))
submission=pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv')
submission['SalePrice']=y_pred
submission.to_csv('submission.csv',index=False)


# Calismamda Fark Ettigim Eksikliklerim ve Not

Bu veri seti secmemdeki amacim aslinda Miuul uzerinden aldigim bir egitimde bu veri setinin incelenmesiydi. Ben de kendim inceleyeyim hem bir calisma yapmis olayim diye basladim ama kaggledaki test seti veri seti ya da direkt VSCode`daki notebookumu aktarinca degiksenlerin adlarini degistirme gibi sorunlari yasadigim icin aslinda analiz etmem gereken bazi seyleri analiz edemedim. Bunlar kolerasyon grafigi, veya ikili degisken analizi. Degisken uretme konusunda kendimi biraz daha zorlayabilirdim sadece sayisal degiskenler icin olan kisma bakip yil farki alan farkli gibi seyleri yaptim, sonra Miuuldaki cozum dosyasinda buldugum degiskenleri karsilastirdim ve ona gore ekledim. Daha farkli degiskenler uretmek icin veriyi daha iyi anlamamam gerekiyor(80 degiksen sayisi beni gercekten zorladi).Ayrica fazladan olan degiskenlerin silinmesi konusundaki cozumum tamamen yapay zekadir. Bu kisimda pek bir bilgim olmadigi icin direkt yapay zekaya danistim. Umarim diger calismamda daha kapsamli analizlere yer verip daha profesyonel bir calisma yapabilirim.
(Ayrica ilk markdown yazilarini olsuturup sonra uzerinde birkac degiisklik yaptigim icin belki kodu degistirip markdowndaki aciklamayi degistirmemis olabilirim.)